# European Soccer Database — Exploration

Source: [hugomathien/soccer on Kaggle](https://www.kaggle.com/datasets/hugomathien/soccer)

Run `python scripts/download_data.py` from the repo root first so that `data/database.sqlite` exists.

This notebook walks through the database structure: it lists the tables, previews a few rows of each, and shows the column schema. It's used as a starting point before building features for the outcome predictor.

In [26]:
import sqlite3
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DB_PATH = REPO_ROOT / "data" / "database.sqlite"
assert DB_PATH.exists(), (
    f"Missing {DB_PATH}. Run `python scripts/download_data.py` from the repo root."
)

conn = sqlite3.connect(DB_PATH)
print(f"Opened {DB_PATH} ({DB_PATH.stat().st_size / 1024**2:.1f} MB)")

Opened /Users/alexy/CSE/Sports-NLP-Outcome-Predictor/data/database.sqlite (298.6 MB)


## Tables in the database

In [27]:
tables = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;",
    conn,
)
tables

,name
0,Country
1,League
2,Match
3,Player
4,Player_Attributes
5,Team
6,Team_Attributes
7,sqlite_sequence


In [28]:
# Row counts per table
row_counts = {
    name: pd.read_sql(f'SELECT COUNT(*) AS n FROM "{name}"', conn)["n"].iloc[0]
    for name in tables["name"]
}
pd.Series(row_counts, name="row_count").sort_values(ascending=False)

Player_Attributes    183978
Match                 25979
Player                11060
Team_Attributes        1458
Team                    299
Country                  11
League                   11
sqlite_sequence           7
Name: row_count, dtype: int64

## Schema and head of every table

In [29]:
from IPython.display import display

for name in tables["name"]:
    try:
        print(f"\n{'=' * 60}")
        print(f"TABLE: {name}")
        print('=' * 60)

        schema = pd.read_sql(f'PRAGMA table_info("{name}");', conn)
        print(f"\n-- Schema ({len(schema)} columns) --")
        display(schema[["name", "type", "notnull", "pk"]])

        row_count = pd.read_sql(f'SELECT COUNT(*) AS n FROM "{name}"', conn)["n"][0]
        print(f"\n-- Head (first 10 of {row_count:,}) --")
        head = pd.read_sql(f'SELECT * FROM "{name}" LIMIT 10', conn)
        display(head)
    except Exception as e:
        print(f"!! Error on {name}: {type(e).__name__}: {e}")


TABLE: Country

-- Schema (2 columns) --


,name,type,notnull,pk
0,id,INTEGER,0,1
1,name,TEXT,0,0



-- Head (first 10 of 11) --


,id,name
0,1,Belgium
1,1729,England
2,4769,France
3,7809,Germany
4,10257,Italy
5,13274,Netherlands
6,15722,Poland
7,17642,Portugal
8,19694,Scotland
9,21518,Spain



TABLE: League

-- Schema (3 columns) --


...

,id,team_fifa_api_id,team_api_id,date,buildUpPlaySpeed,buildUpPlaySpeedClass,buildUpPlayDribbling,buildUpPlayDribblingClass,buildUpPlayPassing,buildUpPlayPassingClass,buildUpPlayPositioningClass,chanceCreationPassing,chanceCreationPassingClass,chanceCreationCrossing,chanceCreationCrossingClass,chanceCreationShooting,chanceCreationShootingClass,chanceCreationPositioningClass,defencePressure,defencePressureClass,defenceAggression,defenceAggressionClass,defenceTeamWidth,defenceTeamWidthClass,defenceDefenderLineClass
0,1,434,9930,2010-02-22 00:00:00,60,Balanced,NaN,Little,50,Mixed,Organised,60,Normal,65,Normal,55,Normal,Organised,50,Medium,55,Press,45,Normal,Cover
1,2,434,9930,2014-09-19 00:00:00,52,Balanced,48.0,Normal,56,Mixed,Organised,54,Normal,63,Normal,64,Normal,Organised,47,Medium,44,Press,54,Normal,Cover
2,3,434,9930,2015-09-10 00:00:00,47,Balanced,41.0,Normal,54,Mixed,Organised,54,Normal,63,Normal,64,Normal,Organised,47,Medium,44,Press,54,Normal,Cover
3,4,77,8485,2010-02-22 00:00:00,70,Fast,NaN,Little,70,Long,Organised,70,Risky,70,Lots,70,Lots,Organised,60,Medium,70,Double,70,Wide,Cover
4,5,77,8485,2011-02-22 00:00:00,47,Balanced,NaN,Little,52,Mixed,Organised,53,Normal,48,Normal,52,Normal,Organised,47,Medium,47,Press,52,Normal,Cover
5,6,77,8485,2012-02-22 00:00:00,58,Balanced,NaN,Little,62,Mixed,Organised,45,Normal,70,Lots,55,Normal,Organised,40,Medium,40,Press,60,Normal,Cover
6,7,77,8485,2013-09-20 00:00:00,62,Balanced,NaN,Little,45,Mixed,Organised,40,Normal,50,Normal,55,Normal,Organised,42,Medium,42,Press,60,Normal,Cover
7,8,77,8485,2014-09-19 00:00:00,58,Balanced,64.0,Normal,62,Mixed,Organised,56,Normal,68,Lots,57,Normal,Organised,41,Medium,42,Press,60,Normal,Cover
8,9,77,8485,2015-09-10 00:00:00,59,Balanced,64.0,Normal,53,Mixed,Organised,51,Normal,72,Lots,63,Normal,Free Form,49,Medium,45,Press,63,Normal,Cover
9,10,614,8576,2010-02-22 00:00:00,60,Balanced,NaN,Little,40,Mixed,Organised,45,Normal,35,Normal,55,Normal,Organised,30,Deep,70,Double,30,Narrow,Offside Trap



TABLE: sqlite_sequence

-- Schema (2 columns) --


,name,type,notnull,pk
0,name,,0,0
1,seq,,0,0



-- Head (first 10 of 7) --


,name,seq
0,Team,103916
1,Country,51958
2,League,51958
3,Match,51958
4,Player,11075
5,Player_Attributes,183978
6,Team_Attributes,1458


## Quick sanity checks

A few starter queries to confirm the join keys behave as expected. Adapt these for feature engineering.

In [30]:
# Matches per league/season
pd.read_sql(
    """
    SELECT l.name AS league, m.season, COUNT(*) AS matches
    FROM Match m
    JOIN League l ON l.id = m.league_id
    GROUP BY l.name, m.season
    ORDER BY l.name, m.season;
    """,
    conn,
).head(20)

,league,season,matches
0,Belgium Jupiler League,2008/2009,306
1,Belgium Jupiler League,2009/2010,210
2,Belgium Jupiler League,2010/2011,240
3,Belgium Jupiler League,2011/2012,240
4,Belgium Jupiler League,2012/2013,240
5,Belgium Jupiler League,2013/2014,12
6,Belgium Jupiler League,2014/2015,240
7,Belgium Jupiler League,2015/2016,240
8,England Premier League,2008/2009,380
9,England Premier League,2009/2010,380


In [31]:
# Goal-difference distribution — useful as a target check for outcome prediction
pd.read_sql(
    """
    SELECT home_team_goal - away_team_goal AS goal_diff, COUNT(*) AS n
    FROM Match
    GROUP BY goal_diff
    ORDER BY goal_diff;
    """,
    conn,
)

,goal_diff,n
0,-9,1
1,-8,3
2,-7,6
3,-6,26
4,-5,85
5,-4,308
6,-3,833
7,-2,2134
8,-1,4070
9,0,6596


In [32]:
conn.close()